# Credit Card Fraud Classification

A focused JupyterLite workflow for rare-event classification. The goal is not raw accuracy; the goal is to choose a threshold that makes false positives and false negatives visible.

## Setup
This notebook uses the small local sample bundled with Pattern Portal so it runs inside JupyterLite. Swap in the full Kaggle dataset in local Jupyter when you want meaningful production-scale results.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = '/cases/datasets/fraud_sample.csv'

def read_portal_csv(path):
    site_path = path if path.startswith('/') else f'/{path}'
    try:
        open_url = __import__('pyodide.http', fromlist=['open_url']).open_url
        return pd.read_csv(open_url(site_path))
    except Exception:
        relative = site_path.lstrip('/')
        candidates = [Path(relative), Path('..') / relative, Path.cwd() / relative, Path.cwd().parent / relative]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate)
        raise FileNotFoundError(f'Could not find {site_path}')

fraud = read_portal_csv(DATA_PATH)
target = 'Class' if 'Class' in fraud.columns else 'is_fraud'
feature_cols = [column for column in fraud.columns if column != target]
print('Rows:', len(fraud))
print('Features:', feature_cols)
display(fraud.head())
display(fraud[target].value_counts().rename('count').to_frame())

In [ ]:
X = fraud[feature_cols].copy()
y = fraud[target].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=RANDOM_STATE,
)

model = RandomForestClassifier(
    n_estimators=120,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
thresholds = np.linspace(0.1, 0.9, 9)
rows = []

for threshold in thresholds:
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    rows.append({
        'threshold': round(float(threshold), 2),
        'precision': round(float(precision), 3),
        'recall': round(float(recall), 3),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
    })

threshold_report = pd.DataFrame(rows)
chosen_threshold = 0.35
chosen_pred = (proba >= chosen_threshold).astype(int)
pr_auc = average_precision_score(y_test, proba)
precision, recall, _ = precision_recall_curve(y_test, proba)

print('Chosen threshold:', chosen_threshold)
print('PR-AUC:', round(float(pr_auc), 4))
print(confusion_matrix(y_test, chosen_pred, labels=[0, 1]))
print(classification_report(y_test, chosen_pred, digits=3, zero_division=0))
display(threshold_report)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, marker='o')
plt.title('Precision-recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.show()